# Attention language model

Start from the same Tiny Shakespeare token stream as the bigram model, then run explicit multi-head causal self-attention on the embedded batch.

In [1]:
import random
import sys
from pathlib import Path

import torch
from torch import nn

repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.dataset import get_batch, load_tiny_shakespeare_tokens, split_token_stream

## Prepare token streams and batches

In [2]:
tokens, vocab, _ = load_tiny_shakespeare_tokens(repo_root / "data")
train_tokens, validation_tokens = split_token_stream(tokens)

vocab_size = len(vocab)
block_size = 8
batch_size = 32
n_embd = 32
num_heads = 2
head_size = n_embd // num_heads

random.seed(42)
x_batch, y_batch = get_batch("train", train_tokens, validation_tokens, block_size, batch_size)
x_batch = torch.tensor(x_batch, dtype=torch.long)
y_batch = torch.tensor(y_batch, dtype=torch.long)

## Embed the current batch

In [3]:
token_embedding_table = nn.Embedding(vocab_size, n_embd)
x = token_embedding_table(x_batch)

assert x.shape == (batch_size, block_size, n_embd)
tuple(x.shape)

(32, 8, 32)

## Multi-head causal self-attention

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, d_model, d_k, d_v):
        super().__init__()
        self.n_head = n_head
        self.d_k = d_k
        self.d_v = d_v

        self.w_qs = nn.Linear(d_model, n_head * d_k, bias=False)
        self.w_ks = nn.Linear(d_model, n_head * d_k, bias=False)
        self.w_vs = nn.Linear(d_model, n_head * d_v, bias=False)
        self.output_projection = nn.Linear(n_head * d_v, d_model, bias=False)

    def forward(self, query, key, value):
        d_k, d_v, n_head = self.d_k, self.d_v, self.n_head
        B, len_q, _ = query.shape
        _, len_k, _ = key.shape
        _, len_v, _ = value.shape

        q = self.w_qs(query).view(B, len_q, n_head, d_k)
        k = self.w_ks(key).view(B, len_k, n_head, d_k)
        v = self.w_vs(value).view(B, len_v, n_head, d_v)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # scaled dot-product attention
        # q @ k^T: (B, h, T, d_k) @ (B, h, d_k, T)
        #          -> (B, h, T, T)
        scores = q @ k.transpose(-2, -1)
        scores = scores / (d_k**0.5)

        # causal mask
        causal_mask = torch.triu(
            torch.ones(len_q, len_k, device=scores.device, dtype=torch.bool), diagonal=1
        )
        scores = scores.masked_fill(causal_mask, float("-inf"))

        # normalize, then retrieve values
        weights = torch.softmax(scores, dim=-1)  # (B, h, T, T)
        output = weights @ v  # (B, h, T, d_v)

        # put heads beside each other again
        output = output.transpose(1, 2).contiguous()  # (B, T, h, d_v)
        output = output.view(B, len_q, n_head * d_v)  # (B, T, h*d_v)
        output = self.output_projection(output)  # (B, T, C)

        return output, weights


mha = MultiHeadAttention(
    n_head=num_heads, d_model=n_embd, d_k=head_size, d_v=head_size
)
out, weights = mha(x, x, x)

assert out.shape == x.shape
assert weights.shape == (batch_size, num_heads, block_size, block_size)
tuple(out.shape)

(32, 8, 32)

## Next step

Wrap this attention module in a minimal transformer block and verify that both residual paths preserve `(B, T, C)`.